# Notebook 3 — Labor Market Demand Analysis

**Project:** Kosovo Labor Market Demand Analysis 2025  
**Description:** This notebook answers seven core research questions and three additional analyses using the ESCO-standardized SuperPuna dataset. All occupation references use the ISCO-08 classification framework.

**Input:** `superpuna_standardized.csv` from Google Drive (produced by Notebook 2)  
**Output:** `Kosovo_Labor_Market_Analysis_2025.xlsx`

**Research questions:**
1. What is the total number of job postings and vacancies?
2. How many unique companies are hiring?
3. How do postings vary over time (day/week/month)?
4. Which companies and occupations post the most?
5. How are vacancies distributed by city and region?
6. What is the average wage by sector?
7. How are vacancies distributed across occupational sectors?

**Additional analyses:**
- Wage distribution histogram
- Work schedule classification (shift duration)
- Company posting activity and returning employers

---

## 1. Install and Import Libraries

In [ ]:
!pip install openpyxl -q

In [ ]:
# Standard library
import os
import re

# Data
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Colab utilities
from google.colab import files, drive

import warnings
warnings.filterwarnings('ignore')

# Chart style
plt.rcParams['figure.dpi']          = 130
plt.rcParams['font.family']         = 'DejaVu Sans'
plt.rcParams['axes.spines.top']     = False
plt.rcParams['axes.spines.right']   = False

print('Libraries loaded successfully.')

## 2. Constants

In [ ]:
TITLE_COL   = 'Titulli_i_vendit_te_punes'
VACANCY_COL = 'Numri_i_vendeve_te_lira'
COMPANY_COL = 'Kompania'
CITY_COL    = 'Vendi_i_punes'
SALARY_COL  = 'Paga'
DATE_COL    = 'Data_e_shpalljes'

ISCO_MAJOR_GROUPS = {
    '1': 'Managers',
    '2': 'Professionals',
    '3': 'Technicians and associate professionals',
    '4': 'Clerical support workers',
    '5': 'Service and sales workers',
    '6': 'Skilled agricultural, forestry and fishery workers',
    '7': 'Craft and related trades workers',
    '8': 'Plant and machine operators and assemblers',
    '9': 'Elementary occupations',
    '0': 'Armed forces occupations'
}

SHORT_SECTOR_NAMES = {
    'Service and sales workers'               : 'Service & Sales',
    'Professionals'                           : 'Professionals',
    'Technicians and associate professionals' : 'Technicians',
    'Managers'                                : 'Managers',
    'Clerical support workers'                : 'Clerical',
}

DRIVE_DIR = '/content/drive/MyDrive/kosovo_lm'

print('Constants defined.')

## 3. Load Standardized Dataset from Google Drive

In [ ]:
drive.mount('/content/drive')

df = pd.read_csv(f'{DRIVE_DIR}/superpuna_standardized.csv')
df[DATE_COL]    = pd.to_datetime(df[DATE_COL], errors='coerce')
df[SALARY_COL]  = pd.to_numeric(df[SALARY_COL], errors='coerce')
df[VACANCY_COL] = pd.to_numeric(df[VACANCY_COL], errors='coerce').fillna(1)
df['month']     = df[DATE_COL].dt.to_period('M')
df['week']      = df[DATE_COL].dt.to_period('W')

print(f'Dataset loaded: {len(df):,} rows x {df.shape[1]} columns')

## 4. Q1 — Total Job Postings and Vacancies by Sector

In [ ]:
total_postings  = len(df)
total_vacancies = int(df[VACANCY_COL].sum())
matched         = df['esco_occupation'].notna().sum()

sector_q1 = (
    df[df['isco_major_group'].notna()]
    .groupby('isco_major_group')
    .agg(Postings=(TITLE_COL, 'count'), Vacancies=(VACANCY_COL, 'sum'))
    .sort_values('Postings', ascending=False)
    .reset_index()
)
sector_q1['Vacancies']   = sector_q1['Vacancies'].astype(int)
sector_q1['% Postings']  = (sector_q1['Postings'] / matched * 100).round(1)

print('=' * 55)
print('     Q1: POSTINGS AND VACANCIES BY SECTOR')
print('=' * 55)
print(f'  Total job postings   : {total_postings:,}')
print(f'  Total vacancies      : {total_vacancies:,}')
print(f'  Matched to ESCO      : {matched:,} ({matched/total_postings*100:.1f}%)')
print(f'\n  {"Sector":<45} {"Postings":>8} {"Vacancies":>10} {"%":>6}')
print('  ' + '-' * 73)
for _, row in sector_q1.iterrows():
    print(f'  {row["isco_major_group"][:45]:<45} '
          f'{int(row["Postings"]):>8,} '
          f'{int(row["Vacancies"]):>10,} '
          f'{row["% Postings"]:>5.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Q1: Job Postings and Vacancies by ISCO Sector — Kosovo 2025',
             fontweight='bold', fontsize=13)

short_labels = [l.split(' and ')[0][:30] for l in sector_q1['isco_major_group']]

bars1 = axes[0].barh(short_labels[::-1], sector_q1['Postings'].values[::-1], color='#003399')
axes[0].set_title('By Job Postings', fontweight='bold')
axes[0].set_xlabel('Number of Postings')
for bar, val in zip(bars1, sector_q1['Postings'].values[::-1]):
    axes[0].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'{int(val):,}', va='center', fontsize=9)

bars2 = axes[1].barh(short_labels[::-1], sector_q1['Vacancies'].values[::-1], color='#FFCC00',
                      edgecolor='#003399', linewidth=0.5)
axes[1].set_title('By Total Vacancies', fontweight='bold')
axes[1].set_xlabel('Number of Vacancies')
for bar, val in zip(bars2, sector_q1['Vacancies'].values[::-1]):
    axes[1].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'{int(val):,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('q1_postings_by_sector.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Q2 — Companies Hiring by Sector

In [ ]:
total_companies  = df[COMPANY_COL].nunique()
cos_by_sector = (
    df[df['isco_major_group'].notna()]
    .groupby('isco_major_group')
    .agg(Companies=(COMPANY_COL, 'nunique'), Postings=(TITLE_COL, 'count'))
    .sort_values('Companies', ascending=False)
    .reset_index()
)

print('=' * 55)
print('          Q2: COMPANIES HIRING BY SECTOR')
print('=' * 55)
print(f'  Total unique companies : {total_companies:,}')
print(f'\n  {"Sector":<45} {"Companies":>10}')
print('  ' + '-' * 57)
for _, row in cos_by_sector.iterrows():
    print(f'  {row["isco_major_group"][:45]:<45} {int(row["Companies"]):>10,}')

print(f'\n  Top 10 companies by postings:')
print(f'  {"Company":<40} {"Postings":>8}   Main Sector')
print('  ' + '-' * 80)
top10 = df.groupby(COMPANY_COL).size().sort_values(ascending=False).head(10)
for company, count in top10.items():
    main_sector = (
        df[df[COMPANY_COL] == company]['isco_major_group']
        .value_counts().index[0]
        if df[df[COMPANY_COL] == company]['isco_major_group'].notna().any()
        else 'Unknown'
    )
    print(f'  {str(company)[:40]:<40} {count:>8,}   {main_sector}')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Q2: Companies Hiring by Sector — Kosovo 2025',
             fontweight='bold', fontsize=13)

short_labels = [l.split(' and ')[0][:30] for l in cos_by_sector['isco_major_group']]
axes[0].barh(short_labels[::-1], cos_by_sector['Companies'].values[::-1], color='#003399')
axes[0].set_title('Unique Companies per Sector', fontweight='bold')
axes[0].set_xlabel('Number of Companies')

top15 = df.groupby(COMPANY_COL).size().sort_values(ascending=False).head(15)
axes[1].barh([str(l)[:30] for l in top15.index[::-1]], top15.values[::-1],
              color='#FFCC00', edgecolor='#003399', linewidth=0.5)
axes[1].set_title('Top 15 Most Active Companies', fontweight='bold')
axes[1].set_xlabel('Number of Postings')

plt.tight_layout()
plt.savefig('q2_companies_by_sector.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Q3 — Postings Over Time

In [ ]:
monthly = df.groupby('month').size()
top5_sectors = (
    df[df['isco_major_group'].notna()]
    .groupby('isco_major_group').size()
    .sort_values(ascending=False).head(5).index.tolist()
)
monthly_by_sector = (
    df[df['isco_major_group'].isin(top5_sectors)]
    .groupby(['month', 'isco_major_group']).size()
    .unstack(fill_value=0)
)
monthly_by_sector.columns = [SHORT_SECTOR_NAMES.get(c, c) for c in monthly_by_sector.columns]

print('=' * 55)
print('         Q3: POSTINGS PER MONTH BY SECTOR')
print('=' * 55)
print(f'  Average postings per month : {monthly.mean():.0f}')
print(f'  Busiest month              : {monthly.idxmax()} ({monthly.max():,} postings)')
print(f'  Quietest month             : {monthly.idxmin()} ({monthly.min():,} postings)')

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Q3: Job Postings Over Time — Kosovo 2025',
             fontweight='bold', fontsize=13)

bars = axes[0].bar(monthly.index.astype(str), monthly.values, color='#003399', width=0.6)
axes[0].set_title('Total Postings per Month', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Postings')
axes[0].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, monthly.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', fontsize=9, fontweight='bold')

colors = ['#003399', '#FFCC00', '#2ecc71', '#9b59b6', '#e74c3c']
for i, col in enumerate(monthly_by_sector.columns):
    axes[1].plot(monthly_by_sector.index.astype(str),
                 monthly_by_sector[col].values,
                 label=col, linewidth=2, marker='o', markersize=4, color=colors[i])
axes[1].set_title('Postings per Month by Top 5 Sectors', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Postings')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('q3_postings_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Q4 — Top Occupations and Companies

In [ ]:
top20_occ = (
    df[df['esco_occupation'].notna()]
    .groupby(['esco_occupation', 'isco_major_group'])
    .agg(Postings=(TITLE_COL, 'count'), Vacancies=(VACANCY_COL, 'sum'))
    .sort_values('Postings', ascending=False)
    .reset_index().head(20)
)
top20_occ['Vacancies'] = top20_occ['Vacancies'].astype(int)

print('=' * 95)
print('              Q4: TOP 20 OCCUPATIONS BY POSTINGS')
print('=' * 95)
print(f'  {"Occupation":<40} {"Sector":<35} {"Postings":>8} {"Vacancies":>10}')
print('  ' + '-' * 97)
for _, row in top20_occ.iterrows():
    print(f'  {str(row["esco_occupation"])[:40]:<40} '
          f'{str(row["isco_major_group"])[:35]:<35} '
          f'{int(row["Postings"]):>8,} '
          f'{int(row["Vacancies"]):>10,}')

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.suptitle('Q4: Top Occupations and Companies — Kosovo 2025',
             fontweight='bold', fontsize=13)

top15_occ = top20_occ.head(15)
axes[0].barh([str(l)[:35] for l in top15_occ['esco_occupation'].values[::-1]],
              top15_occ['Postings'].values[::-1], color='#003399')
axes[0].set_title('Top 15 Occupations by Postings', fontweight='bold')
axes[0].set_xlabel('Number of Postings')

top15_vac = (
    df.groupby(COMPANY_COL)[VACANCY_COL].sum()
    .sort_values(ascending=False).head(15)
)
axes[1].barh([str(l)[:35] for l in top15_vac.index[::-1]], top15_vac.values[::-1],
              color='#FFCC00', edgecolor='#003399', linewidth=0.5)
axes[1].set_title('Top 15 Companies by Total Vacancies', fontweight='bold')
axes[1].set_xlabel('Total Vacancies')

plt.tight_layout()
plt.savefig('q4_top_occupations.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Q5 — Vacancies by City and Sector

In [ ]:
df[CITY_COL] = df[CITY_COL].str.strip()

city_q5 = (
    df.groupby(CITY_COL)
    .agg(Postings=(TITLE_COL, 'count'), Vacancies=(VACANCY_COL, 'sum'))
    .sort_values('Vacancies', ascending=False)
    .reset_index()
)
city_q5['Vacancies']     = city_q5['Vacancies'].astype(int)
city_q5['Avg Vac/Post']  = (city_q5['Vacancies'] / city_q5['Postings']).round(2)

print('=' * 65)
print('            Q5: VACANCIES BY CITY AND SECTOR')
print('=' * 65)
print(f'  {"City":<30} {"Postings":>10} {"Vacancies":>12} {"Avg Vac/Post":>14}')
print('  ' + '-' * 70)
for _, row in city_q5.head(20).iterrows():
    print(f'  {str(row[CITY_COL])[:30]:<30} '
          f'{int(row["Postings"]):>10,} '
          f'{int(row["Vacancies"]):>12,} '
          f'{row["Avg Vac/Post"]:>13.2f}')

# City x Sector heatmap
top8_cities  = city_q5.head(8)[CITY_COL].tolist()
top5_sectors = (
    df[df['isco_major_group'].notna()].groupby('isco_major_group').size()
    .sort_values(ascending=False).head(5).index.tolist()
)
city_sector = (
    df[df[CITY_COL].isin(top8_cities) & df['isco_major_group'].isin(top5_sectors)]
    .groupby([CITY_COL, 'isco_major_group'])[VACANCY_COL].sum()
    .unstack(fill_value=0)
)
city_sector.columns = [SHORT_SECTOR_NAMES.get(c, c[:15]) for c in city_sector.columns]

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.suptitle('Q5: Vacancies by City and Sector — Kosovo 2025',
             fontweight='bold', fontsize=13)

top20 = city_q5.head(20)
axes[0].barh([str(l)[:25] for l in top20[CITY_COL].values[::-1]],
              top20['Vacancies'].values[::-1], color='#003399')
axes[0].set_title('Top 20 Cities by Total Vacancies', fontweight='bold')
axes[0].set_xlabel('Total Vacancies')

heatmap_data = city_sector.values
im = axes[1].imshow(heatmap_data, cmap='Blues', aspect='auto')
axes[1].set_xticks(range(len(city_sector.columns)))
axes[1].set_xticklabels(city_sector.columns, rotation=30, ha='right', fontsize=9)
axes[1].set_yticks(range(len(city_sector.index)))
axes[1].set_yticklabels(city_sector.index, fontsize=9)
axes[1].set_title('Vacancies Heatmap: City x Sector', fontweight='bold')
for i in range(len(city_sector.index)):
    for j in range(len(city_sector.columns)):
        val   = int(heatmap_data[i, j])
        color = 'white' if val > heatmap_data.max() * 0.6 else 'black'
        axes[1].text(j, i, str(val), ha='center', va='center',
                     fontsize=9, color=color, fontweight='bold')
plt.colorbar(im, ax=axes[1], label='Vacancies')
plt.tight_layout()
plt.savefig('q5_vacancies_by_city.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Q6 — Average Wage by Sector

In [ ]:
wage_q6 = (
    df[df['isco_major_group'].notna() & df[SALARY_COL].notna()]
    .groupby('isco_major_group')[SALARY_COL]
    .agg(Count='count', Mean='mean', Median='median', Min='min', Max='max')
    .round(2).sort_values('Mean', ascending=False).reset_index()
)

overall_mean   = df[SALARY_COL].mean()
overall_median = df[SALARY_COL].median()

print('=' * 82)
print('                    Q6: AVERAGE WAGE BY SECTOR')
print('=' * 82)
print(f'  {"Sector":<45} {"Count":>6} {"Mean":>8} {"Median":>8} {"Min":>7} {"Max":>7}')
print('  ' + '-' * 84)
for _, row in wage_q6.iterrows():
    print(f'  {str(row["isco_major_group"])[:45]:<45} '
          f'{int(row["Count"]):>6,} '
          f'{row["Mean"]:>7.0f}EUR '
          f'{row["Median"]:>7.0f}EUR '
          f'{row["Min"]:>6.0f}EUR '
          f'{row["Max"]:>6.0f}EUR')
print(f'\n  Overall mean wage   : {overall_mean:.2f} EUR')
print(f'  Overall median wage : {overall_median:.2f} EUR')

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Q6: Average Wage by Sector — Kosovo 2025',
             fontweight='bold', fontsize=13)

short_labels = [l.split(' and ')[0][:25] for l in wage_q6['isco_major_group']]
x     = range(len(short_labels))
width = 0.35
axes[0].bar([i - width/2 for i in x], wage_q6['Mean'],   width, label='Mean',   color='#003399')
axes[0].bar([i + width/2 for i in x], wage_q6['Median'], width, label='Median', color='#FFCC00',
             edgecolor='#003399', linewidth=0.5)
axes[0].axhline(y=overall_median, color='gray', linestyle='--', linewidth=1,
                label=f'Overall median ({overall_median:.0f} EUR)')
axes[0].set_title('Mean vs Median Wage by Sector', fontweight='bold')
axes[0].set_ylabel('Monthly Wage (EUR)')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(short_labels, rotation=35, ha='right', fontsize=8)
axes[0].legend()

top15_wage = wage_q6.head(15)
axes[1].barh([str(l)[:35] for l in top15_wage['isco_major_group'].values[::-1]],
              top15_wage['Mean'].values[::-1], color='#003399')
axes[1].axvline(x=overall_mean, color='gray', linestyle='--', linewidth=1.5,
                label=f'Overall avg ({overall_mean:.0f} EUR)')
axes[1].set_title('Average Wage by Sector', fontweight='bold')
axes[1].set_xlabel('Average Monthly Wage (EUR)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('q6_wage_by_sector.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Q7 — Vacancy Distribution by Sector

In [ ]:
vac_q7 = (
    df[df['isco_major_group'].notna()]
    .groupby('isco_major_group')
    .agg(Vacancies=(VACANCY_COL, 'sum'), Postings=(TITLE_COL, 'count'),
         Companies=(COMPANY_COL, 'nunique'))
    .sort_values('Vacancies', ascending=False).reset_index()
)
vac_q7['Vacancies']    = vac_q7['Vacancies'].astype(int)
total_vac              = vac_q7['Vacancies'].sum()
vac_q7['% of Total']   = (vac_q7['Vacancies'] / total_vac * 100).round(1)
vac_q7['Avg Vac/Post'] = (vac_q7['Vacancies'] / vac_q7['Postings']).round(2)

print('=' * 82)
print('                Q7: VACANCY DISTRIBUTION BY SECTOR')
print('=' * 82)
print(f'  {"Sector":<45} {"Vacancies":>10} {"% Total":>8} {"Postings":>10} {"Avg/Post":>9}')
print('  ' + '-' * 86)
for _, row in vac_q7.iterrows():
    print(f'  {str(row["isco_major_group"])[:45]:<45} '
          f'{int(row["Vacancies"]):>10,} '
          f'{row["% of Total"]:>7.1f}% '
          f'{int(row["Postings"]):>10,} '
          f'{row["Avg Vac/Post"]:>9.2f}')

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.suptitle('Q7: Vacancy Distribution by Sector — Kosovo 2025',
             fontweight='bold', fontsize=13)

short_labels = [l.split(' and ')[0][:25] for l in vac_q7['isco_major_group']]
colors = ['#003399','#FFCC00','#2ecc71','#9b59b6','#e74c3c',
          '#1abc9c','#e67e22','#34495e','#95a5a6']

axes[0].pie(vac_q7['Vacancies'],
             labels=[f"{l}\n({p}%)" for l, p in zip(short_labels, vac_q7['% of Total'])],
             colors=colors[:len(vac_q7)], startangle=140)
axes[0].set_title('Share of Total Vacancies by Sector', fontweight='bold')

x = range(len(short_labels))
w = 0.35
axes[1].bar([i - w/2 for i in x], vac_q7['Postings'],  w, label='Postings',  color='#003399')
axes[1].bar([i + w/2 for i in x], vac_q7['Vacancies'], w, label='Vacancies', color='#FFCC00',
             edgecolor='#003399', linewidth=0.5)
axes[1].set_title('Postings vs Vacancies by Sector', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(short_labels, rotation=35, ha='right', fontsize=8)
axes[1].legend()

plt.tight_layout()
plt.savefig('q7_vacancy_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Additional Analysis — Wage Distribution

In [ ]:
# A histogram reveals the full shape of the wage distribution,
# which averages alone cannot show. The gap between mean and median
# indicates how much the distribution is skewed by high-paying outliers.

brackets = [0, 350, 400, 500, 600, 700, 1000, 2000]
labels   = ['Up to 350', '351-400', '401-500',
            '501-600',   '601-700', '701-1000', 'Over 1000']
df['wage_bracket'] = pd.cut(df[SALARY_COL], bins=brackets, labels=labels)
bracket_counts     = df['wage_bracket'].value_counts().sort_index()

print('=' * 50)
print('         WAGE DISTRIBUTION')
print('=' * 50)
print(f'  Mean wage          : {df[SALARY_COL].mean():.2f} EUR')
print(f'  Median wage        : {df[SALARY_COL].median():.2f} EUR')
print(f'  Min wage           : {df[SALARY_COL].min():.2f} EUR')
print(f'  Max wage           : {df[SALARY_COL].max():.2f} EUR')
print(f'\n  Wage brackets:')
for label, count in zip(bracket_counts.index, bracket_counts.values):
    print(f'  {label:<15} {count:>6,}  ({count/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Wage Distribution — Kosovo 2025', fontweight='bold', fontsize=13)

axes[0].hist(df[SALARY_COL].dropna(), bins=50, color='#003399',
              edgecolor='white', linewidth=0.5)
axes[0].axvline(df[SALARY_COL].mean(),   color='red',    linestyle='--', linewidth=2,
                label=f'Mean: {df[SALARY_COL].mean():.0f} EUR')
axes[0].axvline(df[SALARY_COL].median(), color='#FFCC00', linestyle='--', linewidth=2,
                label=f'Median: {df[SALARY_COL].median():.0f} EUR')
axes[0].set_title('Distribution of All Wages', fontweight='bold')
axes[0].set_xlabel('Monthly Wage (EUR)')
axes[0].set_ylabel('Number of Postings')
axes[0].legend()

colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#1abc9c','#3498db','#9b59b6']
bars   = axes[1].bar(range(len(bracket_counts)), bracket_counts.values,
                      color=colors, width=0.6)
axes[1].set_xticks(range(len(bracket_counts)))
axes[1].set_xticklabels(bracket_counts.index, rotation=20, ha='right', fontsize=9)
axes[1].set_title('Postings by Wage Bracket (EUR/month)', fontweight='bold')
axes[1].set_ylabel('Number of Postings')
for bar, val in zip(bars, bracket_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('analysis_wage_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Additional Analysis — Shift Duration

In [ ]:
def classify_shift(schedule):
    """
    Parse a time range string ('HH:MM:SS - HH:MM:SS') and classify
    the shift duration into four categories. Overnight shifts are
    handled by adding 24 hours when end time is before start time.
    """
    if not isinstance(schedule, str):
        return 'Unknown'
    try:
        parts   = schedule.strip().split(' - ')
        start_h = int(parts[0][:2]) + int(parts[0][3:5]) / 60
        end_h   = int(parts[1][:2]) + int(parts[1][3:5]) / 60
        if end_h < start_h:
            end_h += 24
        hours = end_h - start_h
        if   hours >= 8: return 'Full-time (8+ hours)'
        elif hours >= 6: return 'Part-time (6-8 hours)'
        elif hours >= 4: return 'Part-time (4-6 hours)'
        else:            return 'Short shift (under 4 hours)'
    except Exception:
        return 'Unknown'

def get_shift_hours(schedule):
    try:
        parts   = schedule.strip().split(' - ')
        start_h = int(parts[0][:2]) + int(parts[0][3:5]) / 60
        end_h   = int(parts[1][:2]) + int(parts[1][3:5]) / 60
        if end_h < start_h:
            end_h += 24
        return round(end_h - start_h, 2)
    except Exception:
        return None


df['shift_type']  = df['Orari_i_punes'].apply(classify_shift)
df['shift_hours'] = df['Orari_i_punes'].apply(get_shift_hours)

shift_dist = df['shift_type'].value_counts()

print('=' * 50)
print('       WORK SCHEDULE — SHIFT DURATION')
print('=' * 50)
for shift, count in shift_dist.items():
    print(f'  {shift:<35} {count:>6,}  ({count/len(df)*100:.1f}%)')
print(f'\n  Average shift length : {df["shift_hours"].mean():.1f} hours')

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Job Postings by Shift Duration — Kosovo 2025',
             fontweight='bold', fontsize=13)

colors = ['#003399', '#FFCC00', '#2ecc71', '#95a5a6']
bars   = ax.bar(range(len(shift_dist)), shift_dist.values,
                 color=colors[:len(shift_dist)], width=0.5)
ax.set_xticks(range(len(shift_dist)))
ax.set_xticklabels(shift_dist.index, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Number of Postings')
for bar, val in zip(bars, shift_dist.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f'{val:,}\n({val/len(df)*100:.1f}%)',
            ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('analysis_shift_duration.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Additional Analysis — Company Posting Activity and Returning Employers

In [ ]:
company_activity = (
    df.groupby(COMPANY_COL)
    .agg(
        first_posting   = (DATE_COL, 'min'),
        last_posting    = (DATE_COL, 'max'),
        total_postings  = (TITLE_COL, 'count'),
        total_vacancies = (VACANCY_COL, 'sum'),
        months_active   = ('month', 'nunique')
    )
    .reset_index()
)
company_activity['days_active'] = (
    company_activity['last_posting'] - company_activity['first_posting']
).dt.days

def classify_poster(months):
    if   months == 1:   return 'One-time (1 month)'
    elif months <= 3:   return 'Occasional (2-3 months)'
    elif months <= 6:   return 'Regular (4-6 months)'
    elif months <= 9:   return 'Frequent (7-9 months)'
    else:               return 'Year-round (10-12 months)'

company_activity['poster_type'] = company_activity['months_active'].apply(classify_poster)

poster_dist      = company_activity['poster_type'].value_counts()
postings_by_type = company_activity.groupby('poster_type')['total_postings'].sum()
total_cos        = len(company_activity)
total_posts      = company_activity['total_postings'].sum()

print('=' * 70)
print('           COMPANY POSTING ACTIVITY AND RETURNING EMPLOYERS')
print('=' * 70)
print(f'\n  Average days active        : {company_activity["days_active"].mean():.0f} days')
print(f'  Companies active 12 months : {(company_activity["months_active"] == 12).sum():,}')
print(f'  Companies active 1 month   : {(company_activity["months_active"] == 1).sum():,}')

order = ['One-time (1 month)', 'Occasional (2-3 months)',
         'Regular (4-6 months)', 'Frequent (7-9 months)', 'Year-round (10-12 months)']

print(f'\n  {"Type":<30} {"Companies":>10} {"% Cos":>7} {"Postings":>10} {"% Posts":>8}')
print('  ' + '-' * 68)
for ptype in order:
    if ptype not in poster_dist.index:
        continue
    cos   = poster_dist[ptype]
    posts = int(postings_by_type.get(ptype, 0))
    print(f'  {ptype:<30} {cos:>10,} {cos/total_cos*100:>6.1f}% '
          f'{posts:>10,} {posts/total_posts*100:>7.1f}%')

print(f'\n  Top 10 companies by days active:')
print(f'  {"Company":<40} {"Days Active":>11} {"Months":>7} {"Postings":>9}')
print('  ' + '-' * 70)
for _, row in company_activity.sort_values('days_active', ascending=False).head(10).iterrows():
    print(f'  {str(row[COMPANY_COL])[:40]:<40} '
          f'{int(row["days_active"]):>11,} '
          f'{int(row["months_active"]):>7} '
          f'{int(row["total_postings"]):>9,}')

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Company Posting Activity — Kosovo 2025',
             fontweight='bold', fontsize=13)

months_dist = company_activity['months_active'].value_counts().sort_index()
axes[0].bar(months_dist.index, months_dist.values, color='#003399', width=0.7)
axes[0].set_title('Companies by Months Active', fontweight='bold')
axes[0].set_xlabel('Months Active')
axes[0].set_ylabel('Number of Companies')
axes[0].set_xticks(range(1, 13))

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#003399']
poster_ordered = {p: poster_dist.get(p, 0) for p in order}
axes[1].bar(range(len(poster_ordered)), list(poster_ordered.values()),
             color=colors, width=0.6)
axes[1].set_xticks(range(len(poster_ordered)))
axes[1].set_xticklabels([o.split('(')[0].strip() for o in order],
                          rotation=20, ha='right', fontsize=9)
axes[1].set_title('Companies by Posting Frequency', fontweight='bold')
axes[1].set_ylabel('Number of Companies')

postings_ordered = {p: int(postings_by_type.get(p, 0)) for p in order}
axes[2].pie(list(postings_ordered.values()),
             labels=[o.split('(')[0].strip() for o in order],
             autopct='%1.1f%%', colors=colors, startangle=140)
axes[2].set_title('Share of Postings by Company Type', fontweight='bold')

plt.tight_layout()
plt.savefig('analysis_company_activity.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Export — ISCO-08 Reference Sheet

This sheet is exported as a standalone Excel file for use as a reference document independent of the analysis notebooks.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = Workbook()
ws = wb.active
ws.title = "ISCO-08 Reference"
ws.sheet_view.showGridLines = False

EU_BLUE        = "003399"
EU_YELLOW      = "FFCC00"
EU_YELLOW_SOFT = "FFF5B8"
EU_WHITE       = "FFFFFF"
EU_DARK        = "1A1A2E"
EU_BORDER      = "CCCCCC"

thin   = Side(style="thin", color=EU_BORDER)
border = Border(top=thin, bottom=thin, left=thin, right=thin)

# Title row
ws.merge_cells("A1:F1")
ws.row_dimensions[1].height = 45
c = ws.cell(1, 1, "ISCO-08 — International Standard Classification of Occupations")
c.font      = Font(name="Arial", bold=True, color=EU_WHITE, size=15)
c.fill      = PatternFill("solid", start_color=EU_BLUE)
c.alignment = Alignment(horizontal="center", vertical="center")

ws.merge_cells("A2:F2")
ws.row_dimensions[2].height = 22
c = ws.cell(2, 1, "Source: International Labour Organization (ILO), 2008 | Used in: Kosovo Labor Market Demand Analysis 2025")
c.font      = Font(name="Arial", italic=True, color=EU_DARK, size=10)
c.fill      = PatternFill("solid", start_color=EU_YELLOW)
c.alignment = Alignment(horizontal="center", vertical="center")

# Column headers
ws.row_dimensions[4].height = 30
headers = ["ISCO Code", "Major Group", "Definition",
           "Key Characteristics", "Education Level", "Examples (Kosovo Data)"]
for col, h in enumerate(headers, 1):
    c = ws.cell(4, col, h)
    c.font      = Font(name="Arial", bold=True, color=EU_WHITE, size=10)
    c.fill      = PatternFill("solid", start_color=EU_BLUE)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border    = border

isco_data = [
    ("1", "Managers",
     "Plan, direct and coordinate the policies and activities of enterprises, governments and other organizations.",
     "Set strategic direction\nManage budgets and staff\nAccountable for results",
     "University degree + significant work experience",
     "Operations Manager, HR Manager, Social Media Manager"),
    ("2", "Professionals",
     "Increase the existing stock of knowledge, apply scientific or artistic concepts and theories.",
     "Apply theoretical knowledge\nConduct research or analysis\nProvide expert advice",
     "University degree (Bachelor, Master or PhD)",
     "Journalist, Architect, Accountant, Software Developer, Doctor"),
    ("3", "Technicians and Associate Professionals",
     "Perform technical tasks connected with research and the application of scientific or artistic concepts.",
     "Apply technical procedures\nOperate complex equipment\nSupport professionals",
     "Post-secondary technical or vocational diploma",
     "IT Support Technician, Administrative Assistant, Dental Assistant"),
    ("4", "Clerical Support Workers",
     "Record, organize, store and retrieve information and perform clerical duties.",
     "Handle correspondence\nManage records and filing\nProcess transactions",
     "Secondary school and on-the-job training",
     "Receptionist, Secretary, Data Entry Clerk, Travel Agent"),
    ("5", "Service and Sales Workers",
     "Provide personal and protective services or sell goods in wholesale or retail trade.",
     "Direct customer interaction\nProvide personal services\nSell products or services",
     "Secondary school or vocational training",
     "Waiter/Waitress, Cashier, Sales Assistant, Security Guard, Cook"),
    ("6", "Skilled Agricultural, Forestry and Fishery Workers",
     "Grow and harvest field and tree crops, breed or hunt animals, catch or culture fish.",
     "Grow crops or breed livestock\nApply agricultural knowledge\nWork seasonally",
     "Primary education and agricultural experience",
     "Farmer, Agricultural Worker, Livestock Breeder"),
    ("7", "Craft and Related Trades Workers",
     "Apply specific knowledge and skills in construction, metal forming and making of precision instruments.",
     "Apply manual trade skills\nConstruct or repair structures\nFollow technical blueprints",
     "Vocational training or apprenticeship",
     "Electrician, Plumber, Carpenter, Welder, Auto Mechanic"),
    ("8", "Plant and Machine Operators and Assemblers",
     "Operate and monitor industrial and agricultural machinery and equipment.",
     "Operate heavy machinery\nMonitor production processes\nDrive vehicles or equipment",
     "Secondary school and technical training",
     "Machine Operator, Truck Driver, Warehouse Worker, Assembler"),
    ("9", "Elementary Occupations",
     "Involve simple and routine tasks which may require the use of hand-held tools and physical effort.",
     "Perform manual or physical tasks\nFollow simple instructions\nEntry-level positions",
     "Primary education or none required",
     "Cleaner, General Labourer, Manufacturing Worker, Packer"),
    ("0", "Armed Forces Occupations",
     "Activities of a military nature carried out by members of the armed forces.",
     "Serve in national defence\nOperate military equipment\nFollow chain of command",
     "Military academy or conscription training",
     "Military Officer, Soldier, Armed Forces NCO"),
]

for i, (code, group, definition, characteristics, education, examples) in enumerate(isco_data):
    r  = i + 5
    bg = EU_YELLOW_SOFT if i % 2 == 0 else EU_WHITE
    ws.row_dimensions[r].height = 70
    for col, val in enumerate([code, group, definition, characteristics, education, examples], 1):
        c = ws.cell(r, col, val)
        c.fill      = PatternFill("solid", start_color=bg)
        c.border    = border
        c.alignment = Alignment(horizontal="center" if col == 1 else "left",
                                 vertical="center", wrap_text=True)
        if col == 1:
            c.font = Font(name="Arial", bold=True, size=16, color=EU_BLUE)
        elif col == 2:
            c.font = Font(name="Arial", bold=True, size=10, color=EU_DARK)
        else:
            c.font = Font(name="Arial", size=9, color=EU_DARK)

for col, width in zip(["A","B","C","D","E","F"], [8, 28, 52, 38, 32, 45]):
    ws.column_dimensions[col].width = width

fname = "ISCO08_Reference_Sheet.xlsx"
wb.save(fname)
files.download(fname)
print(f"ISCO-08 Reference Sheet exported: {fname}")


## 15. Methodological Notes

### Occupation Standardization
Job titles were standardized using a two-stage matching pipeline against the ESCO v1.2 occupation dataset (European Commission, 2024). Stage 1 applied fuzzy string matching (RapidFuzz, token sort ratio, threshold 85/100). Stage 2 applied semantic similarity matching using the `all-MiniLM-L6-v2` sentence transformer model (Reimers & Gurevych, 2019), with a cosine similarity threshold of 0.50. A small number of known mismatches were corrected through manual overrides.

### Translation
Albanian job titles were translated to English using the Google Translate free tier via the `deep-translator` library. Only unique normalized titles were translated. A correction dictionary was applied to fix known translation errors for short Albanian words that lose context when translated in isolation.

### Wage Data
All wage figures are gross monthly wages in EUR as advertised in job postings. The statutory minimum wage in Kosovo during the analysis period was 350 EUR/month.

### References
- European Commission (2024). ESCO v1.2. https://esco.ec.europa.eu
- ILO (2012). ISCO-08. https://www.ilo.org/public/english/bureau/stat/isco/isco08/
- Reimers, N. & Gurevych, I. (2019). Sentence-BERT. EMNLP 2019. https://arxiv.org/abs/1908.10084